# Phiên đo lại: BR-33 (#84) + router dùng chung embedder (#85)

Một phiên Colab GPU, hai phép đo. **Không train, không sinh dữ liệu, không gọi API ngoài.**

| Phần | Issue | Đo gì | Thời gian ước |
|---|---|---|---|
| A | #84 | `Qwen/Qwen3-8B-AWQ` so với `anser-v3-awq`, bốn nhiệm vụ, ctx 32768, so theo cặp (McNemar) | ~60–80 phút |
| B | #85 | router trên event loop + kho tri thức trên worker dùng chung một embedder: ngoại lệ, kết quả lệch, p95 | ~10 phút |

**Trước khi chạy:**
1. Runtime → Change runtime type → **L4 GPU** (hoặc lớn hơn).
2. Trên Drive phải có hai thứ (của chủ dự án, nhờ chia sẻ nếu chưa có; thêm lối tắt vào *My Drive*):
   - thư mục `generated/` chứa `eval_extraction.jsonl`, `eval_narration.jsonl`, `eval_agent.jsonl`, `eval_n8n.jsonl`
   - thư mục model `anser-v3-awq/`
3. Sửa hai đường dẫn ở ô 1.2 nếu Drive của bạn đặt khác.

Các tập đo là câu hư cấu do team viết (#84 mục 3). Không đưa dữ liệu khách thật vào phiên này (R2).
Colab chỉ để đo, không phục vụ traffic thật (AGENTS §3.1).

## 1. Chuẩn bị

In [ ]:
# 1.1 — GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 1.2 — Drive + mã nguồn + kiểm tập đo
import datetime
import os
import shutil
import subprocess
from pathlib import Path

from google.colab import drive

# ==== SỬA HAI DÒNG NÀY NẾU DRIVE CỦA BẠN KHÁC ====
GEN_DIR = Path("/content/drive/MyDrive/ANSER_AI_Logistics/generated")
AWQ_DIR = Path("/content/drive/MyDrive/ANSER_AI_Logistics/anser-v3-awq")
# ===================================================

REPO   = "https://github.com/ANSER-Logistics/ANSER_AI_Logistics.git"
# Script #85 và BENCH_MAX_MODEL_LEN vào nhánh tính năng qua PR "phiên đo BR-33 + #85". Chưa gộp
# thì đặt BRANCH = "feat/84-85-phien-do-colab"; gộp rồi thì để nguyên dòng dưới.
BRANCH = "feat/workflow-format-and-deterministic-core"
WORK   = Path("/content/ANSER_AI")

drive.mount("/content/drive")
if WORK.exists():
    shutil.rmtree(WORK)
r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, str(WORK)],
                   capture_output=True, text=True)
if r.returncode != 0:
    raise SystemExit(f"Không clone được:\n{r.stderr}")
os.chdir(WORK)
SHA = subprocess.run(["git", "log", "-1", "--format=%h"], capture_output=True, text=True).stdout.strip()
print("commit:", subprocess.run(["git", "log", "-1", "--format=%h %ci %s"],
                                capture_output=True, text=True).stdout.strip())

# Phiên 16/11 chạy nhầm nhánh: không có script #85, và ctx thật là 8192 trong khi bản tóm tắt
# in 32768. Chặn ngay ở đây thay vì phát hiện sau một giờ GPU.
_thieu = []
if not (WORK / "offline_training/do_router_dung_chung_embedder.py").exists():
    _thieu.append("offline_training/do_router_dung_chung_embedder.py")
if "BENCH_MAX_MODEL_LEN" not in (WORK / "offline_training/providers.py").read_text(encoding="utf-8"):
    _thieu.append("BENCH_MAX_MODEL_LEN trong providers.py")
if _thieu:
    raise SystemExit(f"Nhánh {BRANCH!r} thiếu: {_thieu}.\n"
                     'PR #106 chưa gộp thì đặt BRANCH = "feat/84-85-phien-do-colab" rồi chạy lại ô này.')

os.environ["ANSER_GENERATED_DIR"] = str(GEN_DIR)
OUT = Path(f"/content/drive/MyDrive/ANSER_AI_Logistics/do_lai_br33_{datetime.date.today():%Y%m%d}_{SHA}")
OUT.mkdir(parents=True, exist_ok=True)
print("kết quả ghi vào:", OUT)

thieu = []
for ten in ("eval_extraction.jsonl", "eval_n8n.jsonl", "eval_narration.jsonl", "eval_agent.jsonl"):
    p = GEN_DIR / ten
    n = sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0
    print(f"  {'✓' if p.exists() else '❌'} {ten:24s} {n if p.exists() else ''}")
    if not p.exists():
        thieu.append(ten)
print(f"  {'✓' if (AWQ_DIR / 'config.json').exists() else '❌'} {AWQ_DIR}")
if thieu:
    print("\n⚠ Thiếu", thieu, "— nhánh đó sẽ in 'KHÔNG ĐO ĐƯỢC'. Theo #84 mục 3: ghi thẳng"
          " 'chưa đo được' vào bảng, KHÔNG đo trên tập khác thay vào.")

In [ ]:
# 1.3 — Thư viện. vLLM 0.10.2 + transformers <5: lý do ở cell 3.1 của ANSER_train_v3_colab.ipynb
# (Python 3.13 trên Colab, GuidedDecodingParams bị gỡ từ 0.12, v5 bỏ all_special_tokens_extended).
# Chạy NGAY SAU 1.2 trong phiên mới: transformers không được import trước khi tắt TensorFlow.
import os

os.environ["USE_TORCH"] = "1"
os.environ["USE_TF"] = "0"
!pip install -q "vllm==0.10.2" "transformers>=4.55.2,<5"
!pip uninstall -y -q tensorflow tf-keras || true
# Phần B (#85) cần kho tri thức thật; cài theo trần đã ghim trong requirements.txt.
# underthesea: BM25 của kho tách từ bằng nó; thiếu thì kho chỉ chạy dense, lệch production (16/11).
!pip install -q -c requirements.txt sentence-transformers chromadb rank_bm25 sentencepiece underthesea
!pip install -q -r requirements-dev.txt -c requirements.txt

import transformers
import vllm
from transformers.utils import is_tf_available

assert not is_tf_available(), "transformers vẫn thấy TensorFlow: Runtime > Restart, chạy lại 1.2 rồi 1.3"
print("vLLM", vllm.__version__, "| transformers", transformers.__version__)

In [ ]:
# 1.4 — Kiểm model AWQ + chép tokenizer chuẩn từ model gốc
# (bộ tokenizer do transformers v5 ghi lúc quantize đọc bằng 4.x thì nổ; xem cell 3.1b notebook train)
import json
import shutil

from huggingface_hub import snapshot_download

cfg = json.loads((AWQ_DIR / "config.json").read_text())
q = cfg.get("quantization_config")
assert q, "config.json thiếu quantization_config — vLLM sẽ nạp như fp16 rồi hết VRAM"
idx = json.loads((AWQ_DIR / "model.safetensors.index.json").read_text())
thieu = [s for s in sorted(set(idx["weight_map"].values())) if not (AWQ_DIR / s).exists()]
assert not thieu, f"thiếu shard {thieu}"
FILES = ["tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt",
         "special_tokens_map.json", "added_tokens.json"]
goc = Path(snapshot_download("Qwen/Qwen3-8B", allow_patterns=FILES))
for ten in FILES:
    if (goc / ten).exists():
        shutil.copy2(goc / ten, AWQ_DIR / ten)
if (AWQ_DIR / "chat_template.jinja").exists():
    (AWQ_DIR / "chat_template.jinja").rename(AWQ_DIR / "chat_template.jinja.bak")
print("✓ AWQ:", q.get("quant_method"), q.get("bits"), "bit — tokenizer đã chép từ Qwen/Qwen3-8B")

## A. #84 — bản gốc so với fine-tune (BR-33)

Cùng commit, cùng phiên, cùng cấu hình: vLLM offline, `BENCH_MAX_MODEL_LEN=32768` (ngữ cảnh production).
Mỗi bản chạy trong một tiến trình riêng (`!python`), nên VRAM được trả lại giữa hai lần.

In [ ]:
# A.1 — Bản gốc (AWQ với AWQ: cùng lượng tử hoá, chỉ khác trọng số)
os.environ["BENCH_MAX_MODEL_LEN"] = "32768"
!python offline_training/benchmark_v3.py --model Qwen/Qwen3-8B-AWQ --no-gate \
    --json /content/goc.json --tho /content/goc_tho.jsonl 2>&1 | tee /content/goc_report.txt

In [ ]:
# A.2 — Bản fine-tune anser-v3 (có cổng: in "dưới ngưỡng" nếu trượt, không dừng notebook)
!python offline_training/benchmark_v3.py --model {AWQ_DIR} \
    --json /content/v3.json --tho /content/v3_tho.jsonl 2>&1 | tee /content/v3_report.txt

In [ ]:
# A.3 — So theo cặp: mỗi nhiệm vụ có "B sửa được", "B LÀM HỎNG", p (McNemar).
!python -m offline_training.compare_runs /content/goc.json /content/v3.json 2>&1 | tee /content/so_cap.txt

In [ ]:
# A.4 — Hai ca grammar thật (cần xgrammar, có sẵn cùng vLLM). Ở máy dev chúng skip.
!python -m pytest tests/unit/agents/test_agentic_plan.py -q -rs 2>&1 | tail -n 15 | tee /content/agentic_plan.txt

## B. #85 — router dùng chung embedder với kho tri thức

Script tự lọc chỉ giữ câu mà router quyết định bằng **embedding** (câu khớp từ khoá không gọi `encode`).
Mã thoát: 0 = không thấy vấn đề, 1 = **có vấn đề**, 2 = không đo được (không phải "đạt").

Đo hai lần: embedder của kho trên **CPU** (mặc định production, `KB_DEVICE=cpu`) và trên **GPU**.

In [ ]:
os.environ["ENV"] = "GPU"
!python offline_training/do_router_dung_chung_embedder.py --luot 3000 --device cpu \
    --out /content/r85_cpu.json 2>&1 | tee /content/r85_cpu.txt
!python offline_training/do_router_dung_chung_embedder.py --luot 3000 --device cuda \
    --out /content/r85_cuda.json 2>&1 | tee /content/r85_cuda.txt

## 2. Lưu kết quả lên Drive + bản tóm tắt để dán vào issue

In [ ]:
import glob

for f in glob.glob("/content/*.json") + glob.glob("/content/*.jsonl") + glob.glob("/content/*.txt"):
    shutil.copy2(f, OUT)
print("Đã lưu", len(list(OUT.iterdir())), "file vào", OUT)

print("\n" + "=" * 70 + "\nDÁN VÀO #84\n" + "=" * 70)
# ctx THẬT SỰ đã dùng, đọc từ dòng providers.py in ra lúc nạp model — không đọc biến môi trường.
import re

for ten in ("goc", "v3"):
    rp = Path(f"/content/{ten}_report.txt")
    m = re.search(r"max_model_len=(\d+)", rp.read_text(encoding="utf-8")) if rp.exists() else None
    print(f"{ten}: max_model_len = {m.group(1) if m else 'KHÔNG THẤY trong báo cáo'}")
print(f"commit {SHA}, vLLM {vllm.__version__}")
print("lệnh: benchmark_v3.py --model Qwen/Qwen3-8B-AWQ --no-gate --json goc.json;"
      " benchmark_v3.py --model anser-v3-awq --json v3.json; compare_runs goc.json v3.json")
print(Path("/content/so_cap.txt").read_text(encoding="utf-8"))

print("=" * 70 + "\nDÁN VÀO #85\n" + "=" * 70)
for d in ("cpu", "cuda"):
    p = Path(f"/content/r85_{d}.txt")
    print(p.read_text(encoding="utf-8") if p.exists() else f"(không có {p})")